In [32]:
import tensorflow as tf 
import keras 

In [19]:
c= tf.Variable([[1,2,3],[4,5,6]] , dtype=float)



In [31]:
indices = [[0, 1], 
           [2, 3]]
values = [5, 10]
dense_shape = [3, 4]

sparse_tensor = tf.SparseTensor(
    indices=indices,
    values=values,
    dense_shape=dense_shape
)

print(sparse_tensor)

SparseTensor(indices=tf.Tensor(
[[0 1]
 [2 3]], shape=(2, 2), dtype=int64), values=tf.Tensor([ 5 10], shape=(2,), dtype=int32), dense_shape=tf.Tensor([3 4], shape=(2,), dtype=int64))


In [33]:
def huber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_loss = tf.square(error) / 2
    linear_loss  = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_loss, linear_loss)

In [34]:
class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold=1.0, **kwargs):
        self.threshold = threshold
        super().__init__(**kwargs)
    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < self.threshold
        squared_loss = tf.square(error) / 2
        linear_loss  = self.threshold * tf.abs(error) - self.threshold**2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)
    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "threshold": self.threshold}

In [35]:
class MyDense(keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)
    def build(self, batch_input_shape):
        self.kernel = self.add_weight(
        name="kernel", shape=[batch_input_shape[-1], self.units],
        initializer="glorot_normal")
        self.bias = self.add_weight(
        name="bias", shape=[self.units], initializer="zeros")
        super().build(batch_input_shape) # must be at the end
    def call(self, X):
        return self.activation(X @ self.kernel + self.bias)
    def compute_output_shape(self, batch_input_shape):
        return tf.TensorShape(batch_input_shape.as_list()[:-1] + [self.units])
    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "units": self.units,
        "activation": keras.activations.serialize(self.activation)}

In [37]:
X_sample = tf.random.normal(shape=(4, 5))
# Create a layer with 3 output neurons
my_layer = MyDense(units=3, activation="relu")
# When the layer receives data for the first time, it calls build() automatically:
output = my_layer(X_sample)

print("\nLayer Weights:")
print("Kernel (Weights) shape:", my_layer.kernel.shape)
print("Bias shape:            ", my_layer.bias.shape)
print("Output shape:          ", output.shape)


Layer Weights:
Kernel (Weights) shape: (5, 3)
Bias shape:             (3,)
Output shape:           (4, 3)


In [39]:
class ResidualBlock(keras.layers.Layer):
    def __init__(self , n_layers , n_neurons , **kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(n_neurons , activation='elu', kernel_initializer='he_normal'
                                          ) for _ in range (n_layers)]
    def __call__(self , inputs):
        x= inputs 
        for layer in self.hidden:
            x= layer(x)
        return inputs + x 

In [40]:
class ResidualRegressor(keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.hidden1 = keras.layers.Dense(30, activation="elu",
        kernel_initializer="he_normal")
        self.block1 = ResidualBlock(2, 30)
        self.block2 = ResidualBlock(2, 30)
        self.out = keras.layers.Dense(output_dim)
    def call(self, inputs):
        Z = self.hidden1(inputs)
        for _ in range(1 + 3):
            Z = self.block1(Z)
        Z = self.block2(Z)
        return self.out(Z)

In [47]:
class ReconstructingRegressor(keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(30, activation="selu",
        kernel_initializer="lecun_normal")
                     for _ in range(5)]
        self.out = keras.layers.Dense(output_dim)
    def build(self, batch_input_shape):
        n_inputs = batch_input_shape[-1]
        self.reconstruct = keras.layers.Dense(n_inputs)
        super().build(batch_input_shape)
    def call(self, inputs):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        reconstruction = self.reconstruct(Z)
        recon_loss = tf.reduce_mean(tf.square(reconstruction - inputs))
        self.add_loss(0.05 * recon_loss)
        return self.out(Z)

In [48]:
l2_reg = keras.regularizers.l2(0.05)
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="elu", kernel_initializer="he_normal",
    kernel_regularizer=l2_reg),
    keras.layers.Dense(1, kernel_regularizer=l2_reg)
])

In [ ]:
import numpy as np 

def random_batch(X, y, batch_size=32):
    idx = np.random.randint(len(X), size=batch_size)
    return X[idx], y[idx]

In [42]:
def print_status_bar(iteration, total, loss, metrics=None):
    metrics = " - ".join(["{}: {:.4f}".format(m.name, m.result())
                    for m in [loss] + (metrics or [])])
    end = "" if iteration < total else "\n"
    print("\r{}/{} - ".format(iteration, total) + metrics,
    end=end)

In [43]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load California housing data
housing = fetch_california_housing()

# 2. Split into train, validation, and test sets
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target.reshape(-1, 1), random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42
)

# 3. Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("y_train shape:       ", y_train.shape)

X_train_scaled shape: (11610, 8)
y_train shape:        (11610, 1)


In [45]:
n_epochs = 5
batch_size = 32
n_steps = len(X_train) // batch_size
optimizer = keras.optimizers.Nadam(learning_rate=0.01)
loss_fn = keras.losses.mean_squared_error

mean_loss = keras.metrics.Mean()
metrics = [keras.metrics.MeanAbsoluteError()]

In [51]:
for epoch in range(1, n_epochs + 1):
    print("Epoch {}/{}".format(epoch, n_epochs))
    for step in range(1, n_steps + 1):
        X_batch, y_batch = random_batch(X_train_scaled, y_train)
        with tf.GradientTape() as tape:
            y_pred = model(X_batch, training=True)
            main_loss = tf.reduce_mean(loss_fn(y_batch, y_pred))
            loss = tf.add_n([main_loss] + model.losses)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        mean_loss(loss)
        for metric in metrics:
            metric(y_batch, y_pred)
        print_status_bar(step * batch_size, len(y_train), mean_loss, metrics)
        print_status_bar(len(y_train), len(y_train), mean_loss, metrics)
  

Epoch 1/5
11610/11610 - mean: 1.2238 - mean_absolute_error: 0.5682
11610/11610 - mean: 1.2229 - mean_absolute_error: 0.5683
11610/11610 - mean: 1.2209 - mean_absolute_error: 0.5681
11610/11610 - mean: 1.2189 - mean_absolute_error: 0.5678
11610/11610 - mean: 1.2178 - mean_absolute_error: 0.5680
11610/11610 - mean: 1.2161 - mean_absolute_error: 0.5679
11610/11610 - mean: 1.2146 - mean_absolute_error: 0.5679
11610/11610 - mean: 1.2133 - mean_absolute_error: 0.5678
11610/11610 - mean: 1.2116 - mean_absolute_error: 0.5676
11610/11610 - mean: 1.2097 - mean_absolute_error: 0.5672
11610/11610 - mean: 1.2085 - mean_absolute_error: 0.5672
11610/11610 - mean: 1.2068 - mean_absolute_error: 0.5668
11610/11610 - mean: 1.2054 - mean_absolute_error: 0.5666
11610/11610 - mean: 1.2043 - mean_absolute_error: 0.5666
11610/11610 - mean: 1.2023 - mean_absolute_error: 0.5662
11610/11610 - mean: 1.2007 - mean_absolute_error: 0.5661
11610/11610 - mean: 1.1993 - mean_absolute_error: 0.5662
11610/11610 - mean: 1

KeyboardInterrupt: 